In [ ]:
# ============================================================
# TFG - ADRIÁN JULVE NAVARRO
# Script completo: scraping + limpieza + análisis de color
# ============================================================

import os
import time
import re
import io
import gzip
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import color as skcolor
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

# Carpeta donde se guarda todo
CARPETA = r"C:\Users\34625\Desktop\4 Carrera\TFG\DATOS SCRAPPING"

CARPETA_MAHOU   = os.path.join(CARPETA, "mahou_data")
CARPETA_ABO     = os.path.join(CARPETA, "abo_data")
CARPETA_OFF     = os.path.join(CARPETA, "openfoodfacts_data")
CARPETA_GRAFICOS = os.path.join(CARPETA, "graficos")

CSV_MAHOU   = os.path.join(CARPETA_MAHOU, "dataset_mahou.csv")
CSV_ABO     = os.path.join(CARPETA_ABO, "dataset_abo.csv")
CSV_OFF     = os.path.join(CARPETA_OFF, "dataset_openfoodfacts.csv")
CSV_FINAL   = os.path.join(CARPETA, "dataset_final_combinado.csv")
CSV_LIMPIO  = os.path.join(CARPETA, "Dataset combinado sin emociones.csv")

# Crear todas las carpetas necesarias
for carpeta in [CARPETA_MAHOU, os.path.join(CARPETA_MAHOU, "imagenes"),
                CARPETA_ABO,   os.path.join(CARPETA_ABO,   "imagenes"),
                               os.path.join(CARPETA_ABO,   "cache"),
                CARPETA_OFF,   os.path.join(CARPETA_OFF,   "imagenes"),
                CARPETA_GRAFICOS]:
    os.makedirs(carpeta, exist_ok=True)


# ============================================================
# FUNCIONES DE COLOR (compartidas por los tres scrapers)
# ============================================================

def descargar_imagen(imagen_url, ruta_destino, headers):
    if os.path.exists(ruta_destino):
        return ruta_destino
    try:
        resp = requests.get(imagen_url, headers=headers, timeout=20)
        img  = Image.open(io.BytesIO(resp.content)).convert("RGB")
        img.save(ruta_destino, "JPEG", quality=90)
        return ruta_destino
    except Exception:
        return None


def calcular_color(ruta):
    img = Image.open(ruta).convert("RGB")
    arr = np.asarray(img, dtype=np.float32)
    mean_R = float(arr[:, :, 0].mean())
    mean_G = float(arr[:, :, 1].mean())
    mean_B = float(arr[:, :, 2].mean())
    lab        = skcolor.rgb2lab(arr / 255.0)
    mean_L     = float(lab[:, :, 0].mean())
    mean_a     = float(lab[:, :, 1].mean())
    mean_b_val = float(lab[:, :, 2].mean())
    contrast_L = float(lab[:, :, 0].std())
    return mean_R, mean_G, mean_B, mean_L, mean_a, mean_b_val, contrast_L


# ============================================================
# PARTE 1 — MAHOU SAN MIGUEL (Selenium)
# ============================================================

def scraper_mahou():
    print("\n" + "="*55)
    print("SCRAPER MAHOU SAN MIGUEL")
    print("="*55)

    HEADERS_MAHOU = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}
    BASE_URL = "https://www.mahou-sanmiguel.com"
    DELAY    = 3
    IMGS_DIR = os.path.join(CARPETA_MAHOU, "imagenes")

    CATEGORIAS = {
        "Cerveza Mahou":        "/tienda/marcas/mahou/producto",
        "Cerveza San Miguel":   "/tienda/marcas/cervezas-san-miguel/producto",
        "Cerveza Alhambra":     "/tienda/marcas/cervezas-alhambra/producto",
        "Corona":               "/tienda/marcas/corona",
        "Founders Brewing":     "/tienda/marcas/founders/producto",
        "Budweiser":            "/tienda/marcas/budweiser",
        "Nómada Brewing":       "/tienda/marcas/nomada",
        "Brutus":               "/tienda/marcas/brutus/producto",
        "Agua Solán de Cabras": "/tienda/marcas/solan-de-cabras/producto",
        "Agua Sierra Natura":   "/tienda/marcas/sierra-natura",
        "Malta y otras bebidas":"/tienda/otras-bebidas",
        "Cristalería y hogar":  "/tienda/regalos-cerveceros/cristaleria-y-hogar",
        "Moda y accesorios":    "/tienda/regalos-cerveceros/moda-y-accesorios",
    }

    # Abrir Chrome en modo invisible
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36")
    opciones.add_argument("--lang=es-ES")
    opciones.add_experimental_option("excludeSwitches", ["enable-logging"])
    servicio = Service(ChromeDriverManager().install())
    driver   = webdriver.Chrome(service=servicio, options=opciones)

    def aceptar_cookies(driver):
        try:
            boton = WebDriverWait(driver, 6).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(translate(., 'ACEPTARCOOKIES', 'aceptarcookies'), 'acepto')]"))
            )
            boton.click()
            time.sleep(1)
        except Exception:
            pass

    def extraer_urls_categoria(driver, path_categoria):
        urls_producto = []
        start = 0
        while True:
            url = f"{BASE_URL}{path_categoria}?start={start}&sz=12"
            driver.get(url)
            time.sleep(DELAY)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
            elementos = driver.find_elements(By.CSS_SELECTOR, "a[href]")
            nuevas = []
            for el in elementos:
                href = el.get_attribute("href") or ""
                if "/tienda/p/" in href and href not in urls_producto and href not in nuevas:
                    nuevas.append(href)
            if not nuevas:
                break
            urls_producto.extend(nuevas)
            start += 12
            if len(nuevas) < 12 or len(urls_producto) >= 300:
                break
        return list(dict.fromkeys(urls_producto))

    def extraer_datos_producto(driver, url_producto, nombre_categoria):
        try:
            driver.get(url_producto)
            time.sleep(DELAY)
            nombre = None
            for selector in ["h1.product-name", "h1.product-detail-name", "h1[itemprop='name']", "h1"]:
                try:
                    el = driver.find_element(By.CSS_SELECTOR, selector)
                    nombre = el.text.strip()
                    if nombre:
                        break
                except Exception:
                    continue
            if not nombre:
                nombre = "Sin nombre"
            imagen_url = None
            try:
                og = driver.find_element(By.XPATH, "//meta[@property='og:image']")
                imagen_url = og.get_attribute("content")
                if imagen_url:
                    imagen_url = re.sub(r'\?.*$', '', imagen_url)
            except Exception:
                pass
            if not imagen_url:
                for selector in [".product-primary-image img", ".product-image img", "img.primary-image", ".product-images img"]:
                    try:
                        img = driver.find_element(By.CSS_SELECTOR, selector)
                        src = (img.get_attribute("data-src") or img.get_attribute("src") or "")
                        if "demandware" in src or "mahou" in src:
                            imagen_url = re.sub(r'\?.*$', '', src)
                            break
                    except Exception:
                        continue
            if not imagen_url:
                return None
            if not imagen_url.startswith("http"):
                imagen_url = BASE_URL + imagen_url
            return {"nombre": nombre, "categoria": nombre_categoria, "imagen_url": imagen_url, "url_producto": url_producto}
        except Exception as e:
            print(f"    Error en {url_producto}: {e}")
            return None

    try:
        driver.get(BASE_URL + "/tienda")
        time.sleep(2)
        aceptar_cookies(driver)

        print("\n[1/3] Recopilando URLs de productos...")
        todos = []
        for nombre_cat, path_cat in CATEGORIAS.items():
            print(f"  {nombre_cat}...", end=" ", flush=True)
            urls = extraer_urls_categoria(driver, path_cat)
            print(f"{len(urls)} productos")
            for url in urls:
                todos.append({"url": url, "categoria": nombre_cat})
            if len(todos) >= 300:
                todos = todos[:300]
                break

        # Eliminar duplicados
        vistos = set()
        todos_unicos = []
        for item in todos:
            if item["url"] not in vistos:
                vistos.add(item["url"])
                todos_unicos.append(item)

        print(f"\n[2/3] Extrayendo datos de {len(todos_unicos)} productos...")
        registros = []
        for item in tqdm(todos_unicos, desc="Scrapeando", unit="prod"):
            datos = extraer_datos_producto(driver, item["url"], item["categoria"])
            if datos:
                registros.append(datos)

    finally:
        driver.quit()

    print(f"\n[3/3] Descargando imágenes y calculando color...")
    filas = []
    for i, reg in enumerate(tqdm(registros, desc="Procesando", unit="img")):
        nombre_limpio  = re.sub(r'[^a-zA-Z0-9]', '_', reg["nombre"])[:40]
        nombre_archivo = f"mahou_{i:04d}_{nombre_limpio}.jpg"
        ruta = descargar_imagen(reg["imagen_url"], os.path.join(IMGS_DIR, nombre_archivo), HEADERS_MAHOU)
        if ruta is None:
            continue
        try:
            mean_R, mean_G, mean_B, mean_L, mean_a, mean_b, contrast_L = calcular_color(ruta)
        except Exception:
            continue
        filas.append({
            "fuente":     "Mahou San Miguel",
            "categoria":  reg["categoria"],
            "nombre":     reg["nombre"],
            "imagen_url": reg["imagen_url"],
            "mean_R":     round(mean_R, 4), "mean_G": round(mean_G, 4), "mean_B": round(mean_B, 4),
            "mean_L":     round(mean_L, 4), "mean_a": round(mean_a, 4), "mean_b": round(mean_b, 4),
            "contrast_L": round(contrast_L, 4),
        })

    df = pd.DataFrame(filas)
    df.to_csv(CSV_MAHOU, index=False, encoding="utf-8-sig")
    print(f"\n✓ Mahou guardado: {len(df)} productos")
    return df


# ============================================================
# PARTE 2 — AMAZON BERKELEY OBJECTS (descarga directa S3)
# ============================================================

def scraper_abo():
    print("\n" + "="*55)
    print("DESCARGADOR AMAZON BERKELEY OBJECTS (ABO)")
    print("="*55)

    HEADERS_ABO = {"User-Agent": "TFG-ColorAnalysis/1.0 (uso-academico)"}
    IMGS_DIR    = os.path.join(CARPETA_ABO, "imagenes")
    CACHE_DIR   = os.path.join(CARPETA_ABO, "cache")
    MAX_PROD    = 10000

    CATEGORIAS_INTERES = {
        "SHIRT", "PANTS", "DRESS", "JACKET", "COAT", "SWEATER", "SKIRT",
        "SHOES", "BOOTS", "SNEAKERS", "SANDALS", "HANDBAG", "BACKPACK",
        "WALLET", "BELT", "WATCH", "SUNGLASSES", "HAT", "SCARF", "GLOVES",
        "JEWELRY", "NECKLACE", "BRACELET", "EARRING", "RING",
        "SOFA", "CHAIR", "TABLE", "BED", "LAMP", "DESK", "SHELF",
        "CABINET", "MIRROR", "RUG", "CURTAIN", "PILLOW", "BLANKET",
        "VASE", "PICTURE_FRAME", "CLOCK",
        "MUG", "CUP", "PLATE", "BOWL", "POT", "PAN", "KNIFE",
        "CUTTING_BOARD", "KETTLE", "TOASTER", "BLENDER", "COFFEE_MAKER",
        "WINE_GLASS", "BOTTLE", "STORAGE_CONTAINER",
        "LAPTOP", "TABLET", "HEADPHONES", "SPEAKER", "CAMERA",
        "KEYBOARD", "MOUSE", "MONITOR", "PHONE_CASE",
        "PERFUME", "LOTION", "SHAMPOO", "TOOTHBRUSH", "HAIR_DRYER",
        "MAKEUP", "LIPSTICK", "NAIL_POLISH",
        "TOY", "PUZZLE", "BOARD_GAME", "DOLL", "ACTION_FIGURE",
        "BOOK", "NOTEBOOK", "PEN",
        "YOGA_MAT", "WATER_BOTTLE", "GYM_BAG", "BIKE", "HELMET",
        "PET_BED", "PET_BOWL", "PET_TOY",
    }

    MAPA_CATEGORIAS = {
        "SHIRT": "Moda - Camisetas", "PANTS": "Moda - Pantalones", "DRESS": "Moda - Vestidos",
        "JACKET": "Moda - Chaquetas", "COAT": "Moda - Abrigos", "SWEATER": "Moda - Jerseys",
        "SKIRT": "Moda - Faldas", "SHOES": "Moda - Zapatos", "BOOTS": "Moda - Botas",
        "SNEAKERS": "Moda - Zapatillas", "SANDALS": "Moda - Sandalias", "HANDBAG": "Moda - Bolsos",
        "BACKPACK": "Moda - Mochilas", "WALLET": "Moda - Carteras", "BELT": "Moda - Cinturones",
        "WATCH": "Moda - Relojes", "SUNGLASSES": "Moda - Gafas de sol", "HAT": "Moda - Sombreros",
        "SCARF": "Moda - Bufandas", "GLOVES": "Moda - Guantes", "JEWELRY": "Moda - Joyería",
        "NECKLACE": "Moda - Collares", "BRACELET": "Moda - Pulseras", "EARRING": "Moda - Pendientes",
        "RING": "Moda - Anillos",
        "SOFA": "Hogar - Sofás", "CHAIR": "Hogar - Sillas", "TABLE": "Hogar - Mesas",
        "BED": "Hogar - Camas", "LAMP": "Hogar - Lámparas", "DESK": "Hogar - Escritorios",
        "SHELF": "Hogar - Estantes", "CABINET": "Hogar - Armarios", "MIRROR": "Hogar - Espejos",
        "RUG": "Hogar - Alfombras", "CURTAIN": "Hogar - Cortinas", "PILLOW": "Hogar - Cojines",
        "BLANKET": "Hogar - Mantas", "VASE": "Hogar - Jarrones", "PICTURE_FRAME": "Hogar - Marcos",
        "CLOCK": "Hogar - Relojes pared",
        "MUG": "Cocina - Tazas", "CUP": "Cocina - Vasos", "PLATE": "Cocina - Platos",
        "BOWL": "Cocina - Boles", "POT": "Cocina - Ollas", "PAN": "Cocina - Sartenes",
        "KNIFE": "Cocina - Cuchillos", "CUTTING_BOARD": "Cocina - Tablas", "KETTLE": "Cocina - Teteras",
        "TOASTER": "Cocina - Tostadoras", "BLENDER": "Cocina - Batidoras", "COFFEE_MAKER": "Cocina - Cafeteras",
        "WINE_GLASS": "Cocina - Copas", "BOTTLE": "Cocina - Botellas", "STORAGE_CONTAINER": "Cocina - Recipientes",
        "LAPTOP": "Tecnología - Portátiles", "TABLET": "Tecnología - Tablets", "HEADPHONES": "Tecnología - Auriculares",
        "SPEAKER": "Tecnología - Altavoces", "CAMERA": "Tecnología - Cámaras", "KEYBOARD": "Tecnología - Teclados",
        "MOUSE": "Tecnología - Ratones", "MONITOR": "Tecnología - Monitores", "PHONE_CASE": "Tecnología - Fundas móvil",
        "PERFUME": "Cosmética - Perfumes", "LOTION": "Cosmética - Cremas", "SHAMPOO": "Cosmética - Champús",
        "TOOTHBRUSH": "Cosmética - Cepillos", "HAIR_DRYER": "Cosmética - Secadores", "MAKEUP": "Cosmética - Maquillaje",
        "LIPSTICK": "Cosmética - Labiales", "NAIL_POLISH": "Cosmética - Esmaltes",
        "TOY": "Juguetes - General", "PUZZLE": "Juguetes - Puzzles", "BOARD_GAME": "Juguetes - Juegos de mesa",
        "DOLL": "Juguetes - Muñecas", "ACTION_FIGURE": "Juguetes - Figuras", "BOOK": "Ocio - Libros",
        "NOTEBOOK": "Ocio - Cuadernos", "PEN": "Ocio - Bolígrafos",
        "YOGA_MAT": "Deporte - Esterillas", "WATER_BOTTLE": "Deporte - Botellas", "GYM_BAG": "Deporte - Bolsas",
        "BIKE": "Deporte - Bicis", "HELMET": "Deporte - Cascos",
        "PET_BED": "Mascotas - Camas", "PET_BOWL": "Mascotas - Comederos", "PET_TOY": "Mascotas - Juguetes",
    }

    def descargar_con_cache(url, nombre_cache):
        ruta_cache = os.path.join(CACHE_DIR, nombre_cache)
        if os.path.exists(ruta_cache):
            print(f"    (usando cache: {nombre_cache})")
            with open(ruta_cache, "rb") as f:
                return f.read()
        print(f"    Descargando {nombre_cache}...")
        resp = requests.get(url, headers=HEADERS_ABO, timeout=60, stream=True)
        resp.raise_for_status()
        datos = resp.content
        with open(ruta_cache, "wb") as f:
            f.write(datos)
        return datos

    def extraer_nombre(listing):
        nombres = listing.get("item_name", [])
        for n in nombres:
            if n.get("language_tag", "").startswith("en"):
                return n.get("value", "Sin nombre").strip()
        if nombres:
            return nombres[0].get("value", "Sin nombre").strip()
        return "Sin nombre"

    def extraer_product_type(listing):
        tipos = listing.get("product_type", [])
        return tipos[0].get("value", "").upper() if tipos else ""

    def extraer_marca(listing):
        marcas = listing.get("brand", [])
        for m in marcas:
            if m.get("language_tag", "").startswith("en"):
                return m.get("value", "").strip()
        if marcas:
            return marcas[0].get("value", "").strip()
        return ""

    # Fase 1: índice de imágenes
    print("\n[1/4] Cargando índice de imágenes...")
    URL_IMAGES_META = "https://amazon-berkeley-objects.s3.amazonaws.com/images/metadata/images.csv.gz"
    datos = descargar_con_cache(URL_IMAGES_META, "images.csv.gz")
    with gzip.open(io.BytesIO(datos), "rt", encoding="utf-8") as f:
        df_imgs = pd.read_csv(f)
    print(f"    → {len(df_imgs)} imágenes en el índice")

    # Fase 2: listings de productos
    print("\n[2/4] Cargando listings de productos...")
    productos_filtrados = []
    URL_LISTINGS_BASE = "https://amazon-berkeley-objects.s3.amazonaws.com/listings/metadata/"

    for num in range(10):  # archivos 0 al 9
        print(f"  Archivo listings_{num}...")
        try:
            url   = URL_LISTINGS_BASE + f"listings_{num}.json.gz"
            datos = descargar_con_cache(url, f"listings_{num}.json.gz")
            with gzip.open(io.BytesIO(datos), "rt", encoding="utf-8") as f:
                for linea in f:
                    try:
                        prod = json.loads(linea.strip())
                    except Exception:
                        continue
                    product_type = extraer_product_type(prod)
                    if product_type in CATEGORIAS_INTERES:
                        main_image_id = prod.get("main_image_id", "")
                        if not main_image_id:
                            continue
                        productos_filtrados.append({
                            "item_id":      prod.get("item_id", ""),
                            "categoria":    MAPA_CATEGORIAS.get(product_type, product_type),
                            "nombre":       extraer_nombre(prod),
                            "marca":        extraer_marca(prod),
                            "main_image_id": main_image_id,
                        })
        except Exception as e:
            print(f"    Error al cargar listings_{num}: {e}")
            continue

        if len(productos_filtrados) >= MAX_PROD:
            break

    # Deduplicar y limitar
    vistos = set()
    unicos = []
    for p in productos_filtrados:
        if p["item_id"] not in vistos:
            vistos.add(p["item_id"])
            unicos.append(p)
    productos_filtrados = unicos[:MAX_PROD]
    print(f"\n  Productos encontrados: {len(productos_filtrados)}")

    # Fase 3: descargar imágenes y calcular color
    print("\n[3/4] Descargando imágenes y calculando color...")
    filas = []
    for i, prod in enumerate(tqdm(productos_filtrados, desc="Procesando", unit="prod")):
        image_id      = prod["main_image_id"]
        imagen_url    = f"https://m.media-amazon.com/images/I/{image_id}._SX512_.jpg"
        nombre_limpio = re.sub(r'[^a-zA-Z0-9]', '_', prod["nombre"])[:35]
        nombre_archivo = f"abo_{i:04d}_{prod['item_id']}_{nombre_limpio}.jpg"
        ruta = descargar_imagen(imagen_url, os.path.join(IMGS_DIR, nombre_archivo), HEADERS_ABO)
        if ruta is None:
            continue
        try:
            mean_R, mean_G, mean_B, mean_L, mean_a, mean_b, contrast_L = calcular_color(ruta)
        except Exception:
            continue
        filas.append({
            "fuente":     "Amazon Berkeley Objects",
            "categoria":  prod["categoria"],
            "nombre":     prod["nombre"],
            "imagen_url": imagen_url,
            "mean_R":     round(mean_R, 4), "mean_G": round(mean_G, 4), "mean_B": round(mean_B, 4),
            "mean_L":     round(mean_L, 4), "mean_a": round(mean_a, 4), "mean_b": round(mean_b, 4),
            "contrast_L": round(contrast_L, 4),
        })

    df = pd.DataFrame(filas)
    df.to_csv(CSV_ABO, index=False, encoding="utf-8-sig")
    print(f"\n✓ ABO guardado: {len(df)} productos")
    return df


# ============================================================
# PARTE 3 — OPEN FOOD FACTS (API pública)
# ============================================================

def scraper_openfoodfacts():
    print("\n" + "="*55)
    print("SCRAPER OPEN FOOD FACTS")
    print("="*55)

    HEADERS_OFF = {"User-Agent": "TFG-ColorAnalysis/1.0 (universidad; uso-academico)"}
    API_BASE    = "https://world.openfoodfacts.org/api/v2/search"
    DELAY       = 1.0
    PAGE_SIZE   = 50
    MAX_PROD    = 10000
    IMGS_DIR    = os.path.join(CARPETA_OFF, "imagenes")

    CATEGORIAS = {
        "Bebidas":              "en:beverages",
        "Agua":                 "en:waters",
        "Zumos":                "en:fruit-juices",
        "Refrescos":            "en:sodas",
        "Cervezas":             "en:beers",
        "Vinos":                "en:wines",
        "Lácteos":              "en:dairy-products",
        "Leche":                "en:milks",
        "Yogures":              "en:yogurts",
        "Quesos":               "en:cheeses",
        "Cereales y desayuno":  "en:breakfast-cereals",
        "Galletas":             "en:biscuits",
        "Chocolates":           "en:chocolates",
        "Snacks":               "en:snacks",
        "Patatas fritas":       "en:chips-and-crisps",
        "Dulces y caramelos":   "en:confectioneries",
        "Conservas":            "en:canned-foods",
        "Salsas":               "en:sauces",
        "Aceites":              "en:oils",
        "Pasta":                "en:pastas",
        "Arroz":                "en:rices",
        "Pan y bollería":       "en:breads",
        "Congelados":           "en:frozen-foods",
        "Carne":                "en:meats",
        "Pescado":              "en:fishes",
        "Frutas":               "en:fruits",
        "Verduras":             "en:vegetables",
        "Legumbres":            "en:legumes",
        "Café e infusiones":    "en:coffees",
        "Condimentos":          "en:condiments",
    }

    CAMPOS_API = ",".join(["code", "product_name", "brands", "categories_tags", "image_front_url", "image_url", "countries_tags"])

    def buscar_productos(categoria_tag, max_productos):
        productos = []
        pagina = 1
        while len(productos) < max_productos:
            params = {
                "categories_tags": categoria_tag,
                "countries_tags":  "en:spain",
                "fields":          CAMPOS_API,
                "page_size":       PAGE_SIZE,
                "page":            pagina,
                "sort_by":         "unique_scans_n",
                "json":            1,
            }
            try:
                resp = requests.get(API_BASE, params=params, headers=HEADERS_OFF, timeout=15)
                resp.raise_for_status()
                datos = resp.json()
            except Exception as e:
                print(f"    Error en API (página {pagina}): {e}")
                break
            items = datos.get("products", [])
            if not items:
                break
            for item in items:
                imagen_url = (item.get("image_front_url") or item.get("image_url") or "")
                if not imagen_url:
                    continue
                nombre = item.get("product_name", "").strip() or "Sin nombre"
                productos.append({"codigo": item.get("code", ""), "nombre": nombre, "marca": item.get("brands", "").strip(), "imagen_url": imagen_url})
            total = datos.get("count", 0)
            if pagina * PAGE_SIZE >= total:
                break
            pagina += 1
            time.sleep(DELAY)
        return productos[:max_productos]

    max_por_categoria = max(10, MAX_PROD // len(CATEGORIAS))

    print(f"\n[1/3] Descargando datos de la API...")
    todos = []
    for nombre_cat, tag_cat in CATEGORIAS.items():
        print(f"  {nombre_cat}...", end=" ", flush=True)
        productos = buscar_productos(tag_cat, max_por_categoria)
        for p in productos:
            p["categoria"] = nombre_cat
        todos.extend(productos)
        print(f"{len(productos)} productos")
        time.sleep(DELAY)

    # Deduplicar
    vistos = set()
    todos_unicos = []
    for p in todos:
        clave = p["codigo"] or p["imagen_url"]
        if clave not in vistos:
            vistos.add(clave)
            todos_unicos.append(p)
    print(f"\n  Total productos únicos: {len(todos_unicos)}")

    print(f"\n[2/3] Descargando imágenes y calculando color...")
    filas = []
    for i, prod in enumerate(tqdm(todos_unicos, desc="Procesando", unit="img")):
        nombre_limpio  = re.sub(r'[^a-zA-Z0-9]', '_', prod["nombre"])[:40]
        codigo_limpio  = re.sub(r'[^a-zA-Z0-9]', '_', prod["codigo"])[:15]
        nombre_archivo = f"off_{i:04d}_{codigo_limpio}_{nombre_limpio}.jpg"
        ruta = descargar_imagen(prod["imagen_url"], os.path.join(IMGS_DIR, nombre_archivo), HEADERS_OFF)
        if ruta is None:
            continue
        try:
            mean_R, mean_G, mean_B, mean_L, mean_a, mean_b, contrast_L = calcular_color(ruta)
        except Exception:
            continue
        filas.append({
            "fuente":     "Open Food Facts",
            "categoria":  prod["categoria"],
            "nombre":     prod["nombre"],
            "imagen_url": prod["imagen_url"],
            "mean_R":     round(mean_R, 4), "mean_G": round(mean_G, 4), "mean_B": round(mean_B, 4),
            "mean_L":     round(mean_L, 4), "mean_a": round(mean_a, 4), "mean_b": round(mean_b, 4),
            "contrast_L": round(contrast_L, 4),
        })

    df = pd.DataFrame(filas)
    df.to_csv(CSV_OFF, index=False, encoding="utf-8-sig")
    print(f"\n✓ Open Food Facts guardado: {len(df)} productos")
    return df


# ============================================================
# PARTE 4 — UNIFICAR LOS TRES DATASETS
# ============================================================

def unificar_datasets():
    print("\n" + "="*55)
    print("UNIFICANDO LOS TRES DATASETS")
    print("="*55)

    COLUMNAS = ["fuente", "categoria", "nombre", "imagen_url",
                "mean_R", "mean_G", "mean_B", "mean_L", "mean_a", "mean_b", "contrast_L"]

    dfs = []
    for path, nombre in [(CSV_ABO, "ABO"), (CSV_MAHOU, "Mahou"), (CSV_OFF, "Open Food Facts")]:
        print(f"  Cargando {nombre}...")
        df = pd.read_csv(path, encoding="utf-8-sig")
        df = df[COLUMNAS]
        print(f"    → {len(df)} filas")
        dfs.append(df)

    df_final = pd.concat(dfs, ignore_index=True)
    df_final = df_final.dropna(subset=["mean_R", "mean_G", "mean_B", "mean_L", "mean_a", "mean_b", "contrast_L"])
    df_final = df_final.drop_duplicates(subset=["imagen_url"])
    df_final = df_final.reset_index(drop=True)

    df_final.to_csv(CSV_FINAL, index=False, encoding="utf-8-sig")
    print(f"\n✓ Dataset combinado: {len(df_final)} productos")
    return df_final


# ============================================================
# PARTE 5 — INGENIERÍA DEL DATO (limpieza + análisis + gráficos)
# ============================================================

def ingenieria_del_dato():
    print("\n" + "="*55)
    print("INGENIERÍA DEL DATO")
    print("="*55)

    df = pd.read_csv(CSV_FINAL, encoding="utf-8-sig")
    print(f"Dataset cargado: {len(df)} filas, {len(df.columns)} columnas")

    variables_color = ["mean_R", "mean_G", "mean_B", "mean_L", "mean_a", "mean_b", "contrast_L"]

    # --- Nulos ---
    print(f"\nValores nulos por columna:")
    print(df.isnull().sum().to_string())
    df = df.dropna(subset=variables_color)

    # --- Duplicados ---
    df = df.drop_duplicates(subset=["imagen_url"])
    df = df.reset_index(drop=True)
    print(f"\nFilas tras limpiar: {len(df)}")

    # --- Validación de rangos ---
    rangos = {"mean_R": (0, 255), "mean_G": (0, 255), "mean_B": (0, 255),
              "mean_L": (0, 100), "mean_a": (-128, 128), "mean_b": (-128, 128), "contrast_L": (0, 100)}
    print("\nValidación de rangos:")
    for var, (vmin, vmax) in rangos.items():
        fuera = ((df[var] < vmin) | (df[var] > vmax)).sum()
        print(f"  {var:<12}: {'✓ OK' if fuera == 0 else f'⚠ {fuera} fuera de rango'}")

    # --- Estadísticos descriptivos ---
    print("\nEstadísticos descriptivos:")
    descripcion = df[variables_color].describe().round(3)
    print(descripcion.to_string())
    descripcion.to_csv(os.path.join(CARPETA_GRAFICOS, "estadisticos_descriptivos.csv"), encoding="utf-8-sig")

    # --- Outliers IQR ---
    print("\nOutliers (método IQR):")
    for var in variables_color:
        Q1 = df[var].quantile(0.25)
        Q3 = df[var].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[var] < Q1 - 1.5*IQR) | (df[var] > Q3 + 1.5*IQR)).sum()
        print(f"  {var:<12}: {outliers} outliers ({round(outliers/len(df)*100, 2)}%)")

    # --- Gráfico distribución por fuente ---
    plt.figure(figsize=(8, 5))
    df["fuente"].value_counts().plot(kind="bar", color=["#2E75B6", "#1F4E79", "#70AD47"])
    plt.title("Número de productos por fuente de datos", fontsize=14, fontweight="bold")
    plt.xlabel("Fuente")
    plt.ylabel("Número de productos")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_GRAFICOS, "distribucion_fuente.png"), dpi=150)
    plt.close()

    # --- Gráfico distribución por categoría ---
    plt.figure(figsize=(12, 6))
    df["categoria"].value_counts().head(15).plot(kind="barh", color="#2E75B6")
    plt.title("Top 15 categorías con más productos", fontsize=14, fontweight="bold")
    plt.xlabel("Número de productos")
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_GRAFICOS, "distribucion_categoria.png"), dpi=150)
    plt.close()

    # --- Histogramas ---
    nombres_variables = {
        "mean_R": "Canal Rojo — RGB (0-255)", "mean_G": "Canal Verde — RGB (0-255)",
        "mean_B": "Canal Azul — RGB (0-255)", "mean_L": "Luminosidad L* — CIELAB (0-100)",
        "mean_a": "Componente a* — CIELAB (-128 a 128)", "mean_b": "Componente b* — CIELAB (-128 a 128)",
        "contrast_L": "Contraste L* — Desv. típica (0-100)",
    }
    colores_hist = ["#C0392B", "#27AE60", "#2980B9", "#8E44AD", "#E67E22", "#16A085", "#2C3E50"]
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle("Distribución de variables de color", fontsize=16, fontweight="bold")
    axes = axes.flatten()
    for i, (var, titulo) in enumerate(nombres_variables.items()):
        axes[i].hist(df[var].dropna(), bins=40, color=colores_hist[i], edgecolor="white", alpha=0.85)
        axes[i].set_title(titulo, fontsize=10, fontweight="bold")
        axes[i].set_xlabel("Valor")
        axes[i].set_ylabel("Frecuencia")
        media = df[var].mean()
        axes[i].axvline(media, color="black", linestyle="--", linewidth=1.2, label=f"Media: {media:.1f}")
        axes[i].legend(fontsize=8)
    axes[7].set_visible(False)
    axes[8].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_GRAFICOS, "histogramas_color.png"), dpi=150)
    plt.close()

    # --- Matriz de correlación ---
    correlacion = df[variables_color].corr().round(2)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, label="Correlación de Pearson")
    ax.set_xticks(range(len(variables_color)))
    ax.set_yticks(range(len(variables_color)))
    ax.set_xticklabels(variables_color, rotation=45, ha="right")
    ax.set_yticklabels(variables_color)
    for i in range(len(variables_color)):
        for j in range(len(variables_color)):
            valor = correlacion.iloc[i, j]
            ax.text(j, i, f"{valor:.2f}", ha="center", va="center",
                    fontsize=9, color="white" if abs(valor) > 0.6 else "black", fontweight="bold")
    ax.set_title("Matriz de correlación — Variables de color", fontsize=13, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_GRAFICOS, "correlacion.png"), dpi=150)
    plt.close()

    # --- Guardar dataset limpio ---
    df.to_csv(CSV_LIMPIO, index=False, encoding="utf-8-sig")
    print(f"\n✓ Dataset limpio guardado: {len(df)} productos, {df['categoria'].nunique()} categorías")
    print(f"  Gráficos guardados en: {CARPETA_GRAFICOS}")

    return df


# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

if __name__ == "__main__":
    print("="*55)
    print("TFG - ADRIÁN JULVE NAVARRO")
    print("Recolección y análisis de color en packaging")
    print("="*55)

    # Cambia a False las partes que ya hayas ejecutado antes
    EJECUTAR_MAHOU = True
    EJECUTAR_ABO   = True
    EJECUTAR_OFF   = True

    if EJECUTAR_MAHOU:
        scraper_mahou()

    if EJECUTAR_ABO:
        scraper_abo()

    if EJECUTAR_OFF:
        scraper_openfoodfacts()

    unificar_datasets()
    ingenieria_del_dato()

    print("\n" + "="*55)
    print("TODO COMPLETADO")
    print("="*55)

